In [0]:
# Databricks notebook source
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.getOrCreate()


In [0]:
silver = spark.table("workspace.default.capstone_silver_sales")

gold = (silver.groupBy("year", "month", "month_name", "state", "category")
        .agg(
            F.countDistinct("order_id").alias("total_orders"),
            F.sum("quantity").alias("units_sold"),
            F.sum("gross_amount").alias("gross_sales"),
            F.sum("discount_amount").alias("discount_amount"),
            F.sum("net_amount").alias("net_sales"),
        )
        .withColumn("avg_order_value", F.round(F.col("net_sales") / F.col("total_orders"), 2)))

In [0]:
gold.write.format("delta").mode("overwrite") \
    .saveAsTable("workspace.default.capstone_gold_sales_summary")

print(f"[GOLD] Summary table created with {gold.count()} rows")